# Game Simulator

In [1]:
# show
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from datetime import date

from common_lib.sql import BigQueryConnector
from common_lib.simulation import (
    SimulationEngine, PlatformInputs,
    save_scenario, load_scenario, list_scenarios,
    save_result, load_result, list_results,
)
from common_lib.widgets import ScenarioPanel
from common_lib.tables import monthly_table, comparison_table, export_all_tables, comparison_table
import datetime as dt

In [2]:
refresh_data = False  # Set to True to refresh data from BigQuery, False to load from local pickle
if refresh_data:
    bqc = BigQueryConnector()

In [3]:
# show

params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}
#cost = bqc.print_cost_estimate('./sql/actuals.sql', is_path=True, query_parameters=actuals_params)

In [4]:
PLATFORM_MAP = {'AND': 'android', 'IOS': 'ios'}
actuals = pd.DataFrame()
if refresh_data:
    
    actuals = bqc.get('./sql/actuals.sql', is_path=True, query_parameters=params)
    pd.to_pickle(actuals, './data/actuals.pkl')
else:
    actuals = pd.read_pickle('./data/actuals.pkl')

actuals['dt'] = pd.to_datetime(actuals['dt'])
actuals['platform'] = actuals['platform'].map(PLATFORM_MAP).fillna(actuals['platform'].str.lower())
actuals = actuals.sort_values('dt')

# Anchor DAU: last observed day per platform
anchor_dau = actuals.sort_values('dt').groupby('platform')['dau'].last().to_dict()

In [5]:
#cost = bqc.print_cost_estimate('./sql/retention.sql',  is_path=True, query_parameters=cohort_params)
#cost = bqc.print_cost_estimate('./sql/conversion.sql', is_path=True, query_parameters=cohort_params)

In [6]:
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}

if refresh_data:
    live_retention  = bqc.get('./sql/retention.sql',  is_path=True, query_parameters=params)
    live_retention.to_pickle('./data/live_retention.pkl')
    live_conversion = bqc.get('./sql/conversion.sql', is_path=True, query_parameters=params)
    live_conversion.to_pickle('./data/live_conversion.pkl')
else:
    live_retention  = pd.read_pickle('./data/live_retention.pkl')
    live_conversion = pd.read_pickle('./data/live_conversion.pkl')

live_retention['platform']  = live_retention['platform'].map(PLATFORM_MAP).fillna(live_retention['platform'].str.lower())
live_conversion['platform'] = live_conversion['platform'].map(PLATFORM_MAP).fillna(live_conversion['platform'].str.lower())

In [7]:
params = {'start_date': '2021-06-01', 'end_date': (date.today() - dt.timedelta(days=1)).strftime('%Y-%m-%d')}

if refresh_data:
    marketing  = bqc.get('./sql/marketing.sql',  is_path=True, query_parameters=params)
    marketing.to_pickle('./data/marketing.pkl')
else:
    marketing  = pd.read_pickle('./data/marketing.pkl')


In [8]:
# CPI and UA spend are loaded on demand via "Get from actuals" buttons in the panel

In [9]:
# show
from common_lib.app import prefill_panel, setup_callbacks

engine = SimulationEngine()
panel  = ScenarioPanel(saved_scenarios=list_scenarios(), last_actuals_date=actuals['dt'].dt.date.max())
prefill_panel(panel, actuals, anchor_dau)
setup_callbacks(panel, engine, actuals,
                live_retention=live_retention, live_conversion=live_conversion,
                installs=actuals, marketing=marketing)
panel.display()

<IPython.core.display.Javascript object>

In [10]:
from common_lib.plots import plot, plot_retention, plot_conversion, configure as configure_plots
from common_lib.simulation import list_results

configure_plots(actuals)

# ── Usage ──────────────────────────────────────────────────────────────────
# plot('base_case')                         # all charts
# plot('base_case', chart='dau')            # DAU only
# plot('base_case', chart='revenue')        # daily revenue
# plot('base_case', chart='monthly')        # monthly bar
# plot(['base_case', 'high_ua'])            # compare scenarios
#
# plot_retention('base_case')               # retention curve from saved scenario
# plot_conversion('base_case')              # conversion curve from saved scenario
# plot_retention(['base_case', 'high_ua'])  # compare retention curves across scenarios
# plot_retention(panel.get_curve_anchors()) # preview current panel state
#print('Available results:', list_results())

In [11]:
#plot('test3', chart='dau')

In [12]:
#plot('test3', chart='revenue')

In [13]:
styled, results = comparison_table(actuals=actuals)
styled


,DAU 2027-03,Cumul Margin 2027-03,vs Target 2027-03,DAU 2027-12,Cumul Margin 2027-12,vs Target 2027-12,DAU 2029-12,Cumul Margin 2029-12,vs Target 2029-12
Scenario,,,,,,,,,
"plan a - 200k 202607, prd uplifts, org uplifts","34,897","$11,122,714",✓ +5.0%,"18,250","$13,781,158",—,"8,589","$17,194,449",—
"plan a - 200k 202607, prd uplifts","31,694","$10,781,700",✓ +1.8%,"17,545","$13,281,836",—,"8,448","$16,602,405",—
"plan a - 200k 202607 50k 2x, prd uplifts, org uplifts","35,322","$11,114,164",✓ +4.9%,"18,377","$13,795,240",—,"8,615","$17,225,242",—
"plan a - 200k 202607 50k x2, prd uplifts","32,119","$10,773,150",✓ +1.7%,"17,672","$13,295,918",—,"8,473","$16,633,198",—
"plan a - 200k 2026decr, prd uplifts, org uplifts","38,026","$11,037,605",✓ +4.2%,"19,046","$13,855,676",—,"8,747","$17,372,865",—
"plan a - 200k 2026decr, prd uplifts","34,823","$10,696,590",✓ +1.0%,"18,341","$13,356,355",—,"8,605","$16,780,821",—
"plan a - no ua, no uplifts","31,694","$10,515,908",✗ -0.7%,"17,545","$12,806,175",—,"8,448","$15,848,275",—


In [14]:
#results.sort_values('Cumul Margin 2027-12', ascending=False)

In [15]:
#export_all_tables(actuals=actuals)

In [16]:
## Things to do next
#- Split IAP from IAA revenue
#- Add an export all / run all function
#- Review and reformulate organic boost uplift
#    - Why is there new installs showing in lesser number that organic uplift
#    - Rethink where the percentage should be applied to
#- 

In [17]:
actuals

,dt,platform,dau,new_installs,iap_revenue,iap_net_revenue,iap_net_factor,ad_revenue,ad_net_revenue
0,2021-06-21,android,5,5,41.465483,29.025838,0.700000,NaN,NaN
1,2021-06-21,ios,4,4,NaN,NaN,NaN,NaN,NaN
2,2021-06-22,android,129,127,NaN,NaN,NaN,NaN,NaN
3,2021-06-22,ios,4,3,NaN,NaN,NaN,NaN,NaN
4,2021-06-23,android,664,610,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...
3709,2026-07-19,ios,43936,799,17149.916008,12894.658706,0.751879,4670.142326,3969.620977
3710,2026-07-20,android,29005,234,8908.158012,6601.735608,0.741089,3023.030634,2569.576039
3711,2026-07-20,ios,43836,802,11981.814387,8985.812571,0.749954,4256.115175,3617.697899
3712,2026-07-21,android,28867,225,7917.285380,5902.415402,0.745510,3128.571053,2659.285395
